# 06.01 章节概述：Conv+BN 算子融合优化

## 学习前置要求

学习本章节前，开发者需要已经具备：

- Python 基础：变量、函数、类、模块导入
- PyTorch 基础：张量操作、nn.Module、自动求导
- MobileNetV3 网络结构：深度可分离卷积、BN、SE、h-swish
- Ascend NPU / CANN 基础：了解 AI 加速器和 torch_npu 的作用

## 章节目标

完成本章节后，你应该能够：

1. 在 MobileNetV3 中定位所有 Conv2d + BatchNorm2d 算子对
2. 推导 BN 折叠公式，理解为什么只适用于推理模式
3. 实现 fuse_conv_bn_eval 与 fuse_model，验证融合前后输出一致
4. 掌握“优化前跑一次、优化后跑一次”的性能对比方法
5. 理解昇腾 CANN / ATC 图编译阶段的算子融合思路

## 章节内容

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">小节</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">Notebook</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">内容</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.2</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.02_experiment_overview.ipynb">实验总览</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">算子优化方法、实验设计与环境检查</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.3</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.03_operator_analysis.ipynb">算子剖析</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">定位 Conv+BN 算子对，推导融合公式</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.4</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.04_baseline_benchmark.ipynb">优化前基线</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">未融合模型推理，保存 baseline.json</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.5</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.05_conv_bn_fusion.ipynb">Conv+BN 融合</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">实现并验证融合，保存 optimized.json</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.6</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.06_comparison_and_conclusion.ipynb">前后对比与结论</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">对比时延、吞吐、显存、算子数量</td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">6.7</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><a href="./06.07_chapter_test.ipynb">章节实践</a></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">综合编程实践题</td>
    </tr>
  </tbody>
</table>

## 环境要求

- CANNLab（Ascend NPU 910B3，32GB HBM）
- Python 3.11.4，PyTorch 2.7.1 + torch_npu
- 依赖：numpy、matplotlib

<div style="text-align: left;">
<img src="./images/operator_optimization_overview.svg" alt="算子优化完整流程" style="display: inline-block;">
</div>